# Utility: Refresh All Lakehouse Monitors

This notebook is used by the `workshop_monitoring_refresh` scheduled job.
It refreshes Lakehouse Monitors for all participant schemas in the `workshop` catalog.

Can also be run manually by the instructor to update all monitoring dashboards.

In [0]:
dbutils.widgets.text("catalog", "workshop", "Catalog")
catalog = dbutils.widgets.get("catalog")
print(f"Refreshing monitors in catalog: {catalog}")

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import MonitorRefreshInfoState

w = WorkspaceClient()

# Find all inference log tables across all schemas in the workshop catalog
# Pattern: workshop.<schema>.churn_inference_log
schemas = w.schemas.list(catalog_name=catalog)

inference_tables = []
for schema in schemas:
    try:
        tables = w.tables.list(catalog_name=catalog, schema_name=schema.name)
        for t in tables:
            if t.name == "churn_inference_log":
                inference_tables.append(t.full_name)
    except Exception:
        pass

print(f"Found {len(inference_tables)} inference log tables:")
for t in inference_tables:
    print(f"  {t}")

In [0]:
refresh_results = []
for table_name in inference_tables:
    try:
        refresh = w.quality_monitors.run_refresh(table_name=table_name)
        refresh_results.append({"table": table_name, "status": "triggered", "refresh_id": refresh.refresh_id})
        print(f"✓ Triggered refresh: {table_name}")
    except Exception as e:
        refresh_results.append({"table": table_name, "status": "error", "error": str(e)})
        print(f"✗ Error refreshing {table_name}: {e}")

print(f"\nRefreshed {sum(1 for r in refresh_results if r['status']=='triggered')}/{len(refresh_results)} monitors")